In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

file_name = "decentralized_simulation"

df = pd.read_csv(f"../results/{file_name}/duel_record.csv")
df = df.sort_values("timestamp")
t0 = df["timestamp"].iloc[0]
df["rel_time"] = df["timestamp"] - t0


nodes = ["node1", "node2", "node3", "node4", "node5", "node6"]
credits = {node: [0.0] for node in nodes}
times = [0.0]

for _, row in df.iterrows():
    for node in nodes:
        credits[node].append(credits[node][-1])
    times.append(row["rel_time"])

    # decrease
    if pd.notna(row["decrease"]):
        for part in row["decrease"].split(";"):
            if part:
                node, val = part.split(":")
                if node in credits:
                    credits[node][-1] -= float(val)
                else:
                    assert node == "node0"
    # increase
    if pd.notna(row["increase"]):
        for part in row["increase"].split(";"):
            if part:
                node, val = part.split(":")
                if node in credits:
                    credits[node][-1] += float(val)
                else:
                    assert node == "node0"


avg_groups = [("node1", "node2"), ("node3", "node4"), ("node5", "node6")]
group_labels = {"node1&node2": "Type A", "node3&node4": "Type B", "node5&node6": "Type C"}

avg_lines = {}

for n1, n2 in avg_groups:
    arr1 = np.array(credits[n1])
    arr2 = np.array(credits[n2])
    avg_lines[group_labels[f"{n1}&{n2}"]] = (arr1 + arr2) / 2


fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [4, 3]})
ax1, ax2 = axes

colors = ["#E15759", "#59A14F", "#4E79A7"]
type_colors = {
    "Type A": "#E15759",
    "Type B": "#59A14F",
    "Type C": "#4E79A7",
}


for (n1, n2), type_key in zip(avg_groups, avg_lines.keys()):
    base_color = type_colors[type_key]
    ax1.plot(times, credits[n1], color=base_color, alpha=0.3, linewidth=0.75)
    ax1.plot(times, credits[n2], color=base_color, alpha=0.3, linewidth=0.75)


for color, (key, arr) in zip(colors, avg_lines.items()):
    ax1.plot(times, arr, color=color, linewidth=2, label=key)

ax1.grid(True, which="both", axis="both", alpha=0.35, linewidth=1)
ax1.axhline(
    y=0,
    color="black",
    linewidth=1.0,
    linestyle="-.",
    alpha=0.8
)


ax1.set_xlabel("Relative Time", fontsize=18)
ax1.set_ylabel("Credit", fontsize=18)


total_requests = {"Type A": 0, "Type B": 0, "Type C": 0}
wins = {"Type A": 0, "Type B": 0, "Type C": 0}
losses = {"Type A": 0, "Type B": 0, "Type C": 0}

def node_group(node):
    for n1, n2 in avg_groups:
        if node in (n1, n2):
            return group_labels[f"{n1}&{n2}"]
    return None

for _, row in df.iterrows():
    # if row["action"] in ["reward", "reward_judge"] and pd.notna(row["increase"]):
    if row["action"] in ["reward"] and pd.notna(row["increase"]):
        for part in row["increase"].split(";"):
            if part:
                node, _ = part.split(":")
                group = node_group(node)
                if group:
                    total_requests[group] += 1

    # transfer_stake：increase=win, decrease=loss
    if row["action"] == "transfer_stake":
        if pd.notna(row["increase"]):
            for part in row["increase"].split(";"):
                if part:
                    node, _ = part.split(":")
                    group = node_group(node)
                    if group:
                        wins[group] += 1

        if pd.notna(row["decrease"]):
            for part in row["decrease"].split(";"):
                if part:
                    node, _ = part.split(":")
                    group = node_group(node)
                    if group:
                        losses[group] += 1

win_rate = {
    g: wins[g] / (wins[g] + losses[g]) if (wins[g] + losses[g]) > 0 else 0.0
    for g in total_requests
}


types = list(avg_lines.keys())
x = np.arange(len(types))
bar_width = 0.25

ax2_count = ax2
bars_total = ax2_count.bar(
    x - bar_width / 2,
    [total_requests[t] for t in types],
    width=bar_width,
    color=[type_colors[t] for t in types],
    alpha=0.35,
    label="Total Requests"
)
ax2_count.set_ylim(0, max(total_requests.values()) * 1.2)

ax2_rate = ax2_count.twinx()
bars_rate = ax2_rate.bar(
    x + bar_width / 2,
    [win_rate[t] for t in types],
    width=bar_width,
    color=[type_colors[t] for t in types],
    alpha=0.9,
    label="Win Rate"
)
ax2_rate.set_ylim(0.0, 1.0)

for rect in bars_total:
    height = rect.get_height()
    ax2_count.text(
        rect.get_x() + rect.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=17,
        color="black",
    )


for rect, t in zip(bars_rate, types):
    height = rect.get_height()
    color = "blue" if height > 0.5 else "red"

    ax2_rate.text(
        rect.get_x() + rect.get_width() / 2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom",
        fontsize=17,
        color=color,
        fontweight="bold"
    )


ax2_rate.axhline(
    0.5,
    linestyle="-.",
    linewidth=1.0,
    color="black",
    alpha=0.5
)

ax2_count.set_xticks(x)
ax2_count.set_xticklabels(["8B", "4B", "0.6B"], fontsize=18, fontweight="bold")  # 16

# Legends
legend_lines = [
    Line2D([0], [0], color=colors[0], lw=3, label="8B"),
    Line2D([0], [0], color=colors[1], lw=3, label="4B"),
    Line2D([0], [0], color=colors[2], lw=3, label="0.6B"),
]

bar_handles = [
    Patch(facecolor="grey", alpha=0.35, label="Total Requests"),
    Patch(facecolor="grey", alpha=0.9, label="Win Rate")
]

fig.legend(
    legend_lines + bar_handles,
    [l.get_label() for l in legend_lines] + [h.get_label() for h in bar_handles],
    loc="upper center",
    ncol=5,
    fontsize=17,
    frameon=False
)

plt.tight_layout(rect=[0,0,1,0.93])

plt.show()